Implementation of the Hattori long-period rotation method (lomb-scargle with offsets)

In [ ]:
import glob
import re
import numpy as np
import pandas as pd
import lightkurve as lk
from scipy.linalg import lstsq
import matplotlib.pyplot as plt


In [ ]:
def load_processed_lcs(path_pattern, bin_minutes=10, sigma_clip=5):
    """
    Load CSV LCs matching tic_<TIC>_s<sector>.csv, preprocess:
      - if median cadence < bin_minutes, bin to bin_minutes
      - remove flux outliers > sigma_clip * σ
    Returns a list of LightCurve sorted by sector.
    """
    files = glob.glob(path_pattern)
    if not files:
        raise FileNotFoundError(f"No files for pattern {path_pattern}")

    # helper: extract numeric sector from filename
    def sector_num(fn):
        m = re.search(r"[sS](\d+)", fn)
        return int(m.group(1)) if m else np.inf

    files_sorted = sorted(files, key=sector_num)
    lcs = []
    bin_size = bin_minutes / 1440.0  # days

    for fn in files_sorted:
        df = pd.read_csv(fn)
        # grab time & flux (first two columns or named)
        if {"time","flux"}.issubset(df.columns):
            t = df["time"].values
            f = df["flux"].values
        else:
            t = df.iloc[:,0].values
            f = df.iloc[:,1].values

        # sort by time
        order = np.argsort(t)
        t, f = t[order], f[order]

        # 1) bin if cadence faster than bin_size
        dt_med = np.median(np.diff(t))
        if dt_med < bin_size:
            edges = np.arange(t.min(), t.max() + bin_size, bin_size)
            inds = np.digitize(t, edges)
            t_b, f_b = [], []
            for i in range(1, len(edges)):
                mask_i = inds == i
                if mask_i.sum() > 0:
                    t_b.append(t[mask_i].mean())
                    f_b.append(f[mask_i].mean())
            t = np.array(t_b)
            f = np.array(f_b)

        # 2) sigma‐clip outliers in this sector
        med = np.median(f)
        std = np.std(f)
        keep = np.abs(f - med) <= sigma_clip * std
        t, f = t[keep], f[keep]

        # build LightCurve
        lc = lk.LightCurve(time=t, flux=f)
        lcs.append(lc)
        print(f"  {fn}: sector s{sector_num(fn):02d}, "
              f"{len(t)} points after binning & clipping")

    return lcs

In [ ]:
def offset_corrected_periodogram(lcs, min_period=0.5, max_period=100, n_periods=5000):
    """
    Returns (periods, powers, offsets, sector_idx, times, fluxes)
    where offsets is array of length M (one per sector).
    """
    # stack
    times = np.hstack([lc.time.value for lc in lcs])
    fluxes = np.hstack([lc.flux for lc in lcs])
    sector_idx = np.hstack([[i]*len(lc.time) for i,lc in enumerate(lcs)])
    # subtract median per sector (subtract or divide???)
    for m in np.unique(sector_idx):
        mask = sector_idx==m
        fluxes[mask] /= np.median(fluxes[mask])
    # step matrix
    M = len(lcs)
    N = len(times)
    step_mat = np.zeros((N, M))
    for m in range(M):
        step_mat[:,m] = (sector_idx==m).astype(float)
    # period grid
    #periods = np.linspace(min_period, max_period, n_periods)
    #periods = np.logspace(np.log10(min_period), np.log10(max_period), num=n_periods)
    # let's do uniform frequency spacing
    freqs = np.linspace(1/min_period,1/max_period,n_periods)
    periods = 1./freqs
    powers = np.zeros_like(periods)
    y = fluxes
    y0 = y - np.mean(y)
    chi2_ref = np.sum(y0**2)
    # LS loop
    for j,P in enumerate(periods):
        ω = 2*np.pi/P
        X = np.column_stack([
            np.sin(ω*times),
            np.cos(ω*times),
            step_mat
        ])
        coeffs, *_ = lstsq(X, y)
        resid = y - X.dot(coeffs)
        powers[j] = (chi2_ref - np.sum(resid**2)) / chi2_ref
    # best‐fit coefficients at P_best
    best_idx = np.argmax(powers)
    ω_best = 2*np.pi/periods[best_idx]
    Xb = np.column_stack([
        np.sin(ω_best*times),
        np.cos(ω_best*times),
        step_mat
    ])
    coeffs_b, *_ = lstsq(Xb, y)
    offsets = coeffs_b[2:]  # one per sector
    return periods, powers, offsets, sector_idx, times, fluxes

In [ ]:
# cell: process all TICs, pick out best & 2nd‐best peaks, save summary CSV

import glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

results = []

tics = [198459831,198459831, 165552443] ## these are in the Hattori paper, confirm we can reproduce
for tic in tics:
    lcs = load_processed_lcs(f"LCs/tic_{tic}_s*.csv")
    nsec = len(lcs)
    # if nsec < 2:
    #     print(f"TIC {tic}: only {nsec} sectors, skipping.")
    #     continue

    # choose max_period
    if   nsec < 3:  maxP = 12
    elif nsec < 4:  maxP = 20
    elif nsec < 5:  maxP = 30
    elif nsec < 6:  maxP = 40
    elif nsec < 8:  maxP = 50
    else:           maxP = 100


    # run periodogram
    periods, powers, offsets, sector_idx, times, fluxes = \
        offset_corrected_periodogram(lcs, min_period=0.2, max_period=maxP)

    # pick best peak
    best_idx  = np.argmax(powers)
    bestP     = periods[best_idx]
    best_pow  = powers[best_idx]

    # pick 2nd best >10% away in period
    mask = np.abs(periods - bestP) > 0.1 * bestP
    if mask.any():
        # find highest power among masked
        sec_rel_idx = np.argmax(powers[mask])
        sec_idx = np.where(mask)[0][sec_rel_idx]
        secondP    = periods[sec_idx]
        second_pow = powers[sec_idx]
    else:
        secondP, second_pow = np.nan, np.nan

    print(f"TIC {tic}: best P={bestP:.3f} d (power {best_pow:.3f}), "
          f"2nd P={secondP:.3f} d (power {second_pow:.3f})")

    # record for CSV
    results.append({
        'TIC': int(tic),
        'period_min': 0.5,
        'period_max': maxP,
        'n_sectors': nsec,
        'best_period': bestP,
        'best_power': best_pow,
        'second_period': secondP,
        'second_power': second_pow
    })

    # — LS power vs period plot —
    plt.figure(figsize=(6,4))
    plt.plot(periods, powers)
    plt.axvline(bestP,   ls="--")
    plt.axvline(secondP, ls="--")
    plt.xlabel("Period [d]")
    plt.ylabel("LS Power")
    plt.title(f"TIC {tic} LS periodogram")
    plt.tight_layout()
    plt.savefig(f"TIC_{tic}_LSpower.pdf")
    plt.close()

    # — phased plots for best & 2nd P —
    fig, axes = plt.subplots(1,2, figsize=(10,4), sharey=True)
    for ax, (P, pow_val, lbl) in zip(axes,
                                     [(bestP,best_pow,"Best"),
                                      (secondP,second_pow,"2nd")]):
        phase = (times % P)/P
        cycle = np.floor((times - times.min())/P)
        y = fluxes + offsets[sector_idx]
        ql, qh = np.quantile(y, [0.001, 0.999])

        sc = ax.scatter(phase, y, c=cycle, s=2,alpha=0.5)
        ax.set_xlim(0,1)
        ax.set_ylim(ql, qh)
        ax.set_xlabel("Phase")
        ax.set_title(f"{lbl} P={P:.3f} d  (power {pow_val:.3f})")

    axes[0].set_ylabel("Flux + sector offset")
    fig.tight_layout()
    fig.savefig(f"TIC_{tic}_phased.pdf")
    plt.close()

# after loop: write out summary CSV
df = pd.DataFrame(results,
    columns=['TIC','period_min','period_max','n_sectors',
             'best_period','best_power','second_period','second_power'])
df.to_csv("tic_summary.csv", index=False)
df  # display in the notebook